In [1]:
# 一、导入库
import pandas as pd
import numpy as np
from sklearn.preprocessing import (
    StandardScaler
)
from sklearn.decomposition import (
    PCA,
    TruncatedSVD
)
from sklearn.feature_extraction.text import (
    TfidfVectorizer
)
# 二、读取预处理后数据
df = pd.read_csv(
    "ADHD_预处理后数据.csv"
)
print("数据量：", len(df))
# 三、构建高被引标签
# （监督学习主标签）
threshold = df[
    "cited_by_count"
].quantile(0.9)

df["is_highly_cited"] = (
    df["cited_by_count"]
    >= threshold
).astype(int)

print(
    "\n高被引阈值：",
    threshold
)
# 四、数值特征工程
print("\n开始数值特征工程...")
# 1. Log变换（解决右偏问题）
df["log_cited_by_count"] = np.log1p(
    df["cited_by_count"]
)
df["log_recent_citations"] = np.log1p(
    df["recent_citations"]
)
df["log_referenced_works"] = np.log1p(
    df["referenced_works_count"]
)
df["log_abstract_length"] = np.log1p(
    df["abstract_length"]
)
# 2. 合作强度特征
# 作者-机构比例
df["author_institution_ratio"] = (
    df["authors_count"]
    / (df["institutions_count"] + 1)
)
# 机构-国家比例
df["institution_country_ratio"] = (
    df["institutions_count"]
    / (df["countries_count"] + 1)
)
# 国际合作强度
df["collaboration_intensity"] = (
    df["authors_count"]
    * df["countries_count"]
)
# 作者平均机构覆盖
df["institution_per_author"] = (
    df["institutions_count"]
    / (df["authors_count"] + 1)
)
# 3. 文献质量特征
# 单位参考文献密度
df["reference_density"] = (
    df["referenced_works_count"]
    / (df["abstract_length"] + 1)
)
# 单位摘要引用率
df["citation_density"] = (
    df["cited_by_count"]
    / (df["abstract_length"] + 1)
)
# 4. 时间特征
current_year = 2025

df["paper_age"] = (
    current_year
    - df["publication_year"]
)
# 年均引用
df["citation_per_year"] = (
    df["cited_by_count"]
    / (df["paper_age"] + 1)
)
# 五、异常值处理（Winsorize思想）
# 防止极端值影响模型
print("\n开始异常值处理...")

numeric_clip_cols = [
    "authors_count",
    "institutions_count",
    "countries_count",
    "referenced_works_count",
    "cited_by_count",
    "recent_citations"
]
for col in numeric_clip_cols:
    q1 = df[col].quantile(0.01)
    q99 = df[col].quantile(0.99)
    df[col] = df[col].clip(
        q1,
        q99
    )
# 六、标准化数值特征
print("\n开始标准化...")
numeric_features = [
    "authors_count",
    "institutions_count",
    "countries_count",
    "concepts_count",
    "topics_count",
    "title_length",
    "abstract_length",
    "referenced_works_count",
    "recent_citations",
    "citation_percentile",
    "log_recent_citations",
    "log_referenced_works",
    "log_abstract_length",
    "author_institution_ratio",
    "institution_country_ratio",
    "collaboration_intensity",
    "institution_per_author",
    "reference_density",
    "citation_density",
    "citation_per_year"
]
# 保留存在字段
numeric_features = [
    col for col in numeric_features
    if col in df.columns
]
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[numeric_features] = scaler.fit_transform(
    df[numeric_features]
)
# 七、文本特征工程
print("\n开始TF-IDF文本特征工程...")

# 1. 合并文本
df_scaled["text_for_cluster"] = (
    df_scaled["title_processed"]
    .fillna("")
    + " "
    + df_scaled["abstract_processed"]
    .fillna("")
    + " "
    + df_scaled["concepts"]
    .fillna("")
    + " "
    + df_scaled["topics"]
    .fillna("")
)
# 2. TF-IDF向量化
tfidf = TfidfVectorizer(
    max_features=1000,
    stop_words="english",
    min_df=5,
    max_df=0.8
)
X_tfidf = tfidf.fit_transform(
    df_scaled["text_for_cluster"]
)
print(
    "TF-IDF矩阵维度：",
    X_tfidf.shape
)
# 八、文本降维
print("\n开始文本降维...")

svd = TruncatedSVD(
    n_components=50,
    random_state=42
)

X_text_svd = svd.fit_transform(
    X_tfidf
)

print(
    "SVD降维后维度：",
    X_text_svd.shape
)

# 累积解释方差
explained = np.sum(
    svd.explained_variance_ratio_
)

print(
    "累计解释方差：",
    round(explained, 4)
)

# 九、数值特征PCA
print("\n开始数值PCA降维...")

pca = PCA(
    n_components=0.90
)

X_numeric_pca = pca.fit_transform(
    df_scaled[numeric_features]
)

print(
    "PCA后维度：",
    X_numeric_pca.shape
)

print(
    "PCA累计解释方差：",
    round(
        np.sum(
            pca.explained_variance_ratio_
        ),
        4
    )
)

# 十、保存建模数据
print("\n开始保存建模数据...")

# 保存完整特征工程数据
df_scaled.to_csv(
    "ADHD_特征工程后数据.csv",
    index=False,
    encoding="utf-8-sig"
)

# 保存SVD文本特征
svd_df = pd.DataFrame(
    X_text_svd
)

svd_df.to_csv(
    "ADHD_SVD文本特征.csv",
    index=False,
    encoding="utf-8-sig"
)

# 保存PCA数值特征
pca_df = pd.DataFrame(
    X_numeric_pca
)

pca_df.to_csv(
    "ADHD_PCA数值特征.csv",
    index=False,
    encoding="utf-8-sig"
)

# 输出结果
print("\n特征工程完成！")
print("\n已生成文件：")
print(
    "1. ADHD_特征工程后数据.csv"
)
print(
    "2. ADHD_SVD文本特征.csv"
)
print(
    "3. ADHD_PCA数值特征.csv"
)
print("\n最终数值特征数量：")
print(len(numeric_features))

数据量： 6551

高被引阈值： 8.0

开始数值特征工程...

开始异常值处理...

开始标准化...

开始TF-IDF文本特征工程...
TF-IDF矩阵维度： (6551, 1000)

开始文本降维...
SVD降维后维度： (6551, 50)
累计解释方差： 0.2857

开始数值PCA降维...
PCA后维度： (6551, 11)
PCA累计解释方差： 0.9036

开始保存建模数据...

特征工程完成！

已生成文件：
1. ADHD_特征工程后数据.csv
2. ADHD_SVD文本特征.csv
3. ADHD_PCA数值特征.csv

最终数值特征数量：
20
